In [26]:
import numpy as np

#####Encoding the dataset

In [27]:
data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0,  2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5,  1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5,  2.3, 0, 'Beer']
]

l= {'Wine':0,'Beer':1,'Whiskey':2}
reverse = {v:k for v,k in l.items()}

x = np.array([[row[0],row[1],row[2]] for row in data])
y = np.array([l[row[3]] for row in data], dtype=int)

#####Gini Impurity

In [28]:
def gini_impurity(labels):
  n = len(labels)
  if(n==0):
    return 0
  classes,counts = np.unique(labels,return_counts = True)
  prob = counts/n
  return 1.0 - (np.sum(prob**2))

#####Best split finder

In [29]:
def best_split(x,y):
  best_feature = None
  best_threshold = None
  best_gini = float('inf')
  n_samples,n_features = x.shape

  for feature in range(n_features):
    thresholds = np.unique(x[:,feature])

    for threshold in thresholds:
      left_mask = x[:, feature] <= threshold
      right_mask = ~left_mask

      y_left = y[left_mask]
      y_right = y[right_mask]

      if len(y_left)==0 or len(y_right)==0:
        continue

      weighted_gini = (len(y_left)/n_samples)* gini_impurity(y_left) +(len(y_right)/n_samples)*gini_impurity(y_right)

      if weighted_gini < best_gini:
        best_gini = weighted_gini
        best_feature = feature
        best_threshold = threshold

  return best_feature,best_threshold


#####Recursive Tree Building

In [30]:
class Node:
    def __init__(self, feature_index = None ,threshold = None,left = None,right =None,value = None):
        self.feature_index = feature_index
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value


def build_tree(x,y,depth = 0,max_depth = 10, min_samples = 1):
    n_samples = len(x)
    if(len(np.unique(y))==1 or n_samples<= min_samples or depth >= max_depth):
      majority_class = np.bincount(y).argmax()
      return Node(value=majority_class)

    feature_index, threshold = best_split(x, y)

    if feature_index is None:
      majority_class = np.bincount(y).argmax()
      return Node(value=majority_class)

    left_mask  = x[:, feature_index] <= threshold
    right_mask = ~left_mask

    left_subtree  = build_tree(x[left_mask],  y[left_mask],  depth + 1, max_depth, min_samples)
    right_subtree = build_tree(x[right_mask], y[right_mask], depth + 1, max_depth, min_samples)

    return Node(feature_index=feature_index, threshold=threshold,left=left_subtree, right=right_subtree)


#####Prediction

In [31]:
from IPython.utils.text import re
def prediction(node,x):
  if node.value is not None:
    return node.value
  if x[node.feature_index] <= node.threshold:
    return prediction(node.left, x)
  else:
    return prediction(node.right, x)

def predict(tree,x):
  return np.array([prediction(tree, x) for x in x])

#####Evaluation

In [33]:
tree = build_tree(x, y, max_depth=10, min_samples=1)

y_pred = predict(tree, x)
accuracy = np.mean(y_pred == y)
print(f"Training Accuracy: {accuracy * 100:.1f}%\n")

test_data = np.array(
    [[6.0,  2.1,  0],
     [39.0, 0.05, 1],
     [13.0, 1.3,  1]]
)

predictions = predict(tree, test_data)
expected    = ['Beer', 'Whiskey', 'Wine']

correct_reverse = {value: key for key, value in l.items()}

print("Test Predictions:")
for i, (pred, exp) in enumerate(zip(predictions, expected)):
  result = "Yes" if correct_reverse[pred] == exp else "No"
  print(f"  Sample {i+1}: Predicted = {correct_reverse[pred]:<8} | Expected = {exp:<8} {result}")

Training Accuracy: 100.0%

Test Predictions:
  Sample 1: Predicted = Wine     | Expected = Beer     No
  Sample 2: Predicted = Whiskey  | Expected = Whiskey  Yes
  Sample 3: Predicted = Wine     | Expected = Wine     Yes
